# Approach 1 (Avoid-Step Cutoff) — Averaged CE vs Fitted Curve

For every (P%, BS) combination: scatter shows averaged raw CE **up to the detected step cutoff**,
overlaid with the smooth fitted curve `A + B/(BN+1)^n` fit on that clean pre-step data.

The cutoff is adaptive per (P%, BS): `cutoff_BN` = first BN ≥ 100 where
`abs(CE(BN) − CE(BN−1)) > 0.01`. If no step detected, the full data range is shown.

Parameters (including `cutoff_BN`) come from
`approach_1_avoid_step/intermediate/approach_1_fit_params_bs_{bs}.csv`.

PNGs saved to `BS_{bs}/fitting_avg_plot_A_1_avoid_step_p_{p}_bs_{bs}.png`.

In [10]:
# === Cell 1 — Config, imports ===
import os, glob, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ── CONFIG ────────────────────────────────────────────────────────────────────────────
# cutoff_BN is read per-row from the intermediate CSV; no fixed BN_MAX needed.
BN_STEP_MIN = 100    # informational only
STEP_THRESH = 0.01
# ───────────────────────────────────────────────────────────────────────────────

BASE_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
INTER_DIR = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test\approach_1_avoid_step\intermediate"
OUT_DIR   = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test\approach_1_avoid_step\avg_plot_v_fitting_curve_avoid_step"

BATCH_SIZES = [64, 1024, 60000]
CE_o = np.log(10)   # ln(10) \u2248 2.302585

p_dirs = glob.glob(os.path.join(BASE_DIR, "p-percentage_*"))
PRUNING_LEVELS = sorted([
    float(re.search(r"p-percentage_([\d.]+)", d).group(1))
    for d in p_dirs
])
print(f"Found {len(PRUNING_LEVELS)} pruning levels: {PRUNING_LEVELS}")
print(f"CE_o = ln(10) = {CE_o:.6f}")
print("Cell 1 ready.")

Found 19 pruning levels: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.82, 0.84, 0.86, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 1.0]
CE_o = ln(10) = 2.302585
Cell 1 ready.


In [11]:
# === Cell 2 — Load all (P%, BS) averaged data and avoid-step fit parameters ===
records = []   # one dict per (p, bs)

for bs in BATCH_SIZES:
    params_csv = os.path.join(INTER_DIR, f"approach_1_fit_params_bs_{bs}.csv")
    if not os.path.exists(params_csv):
        print(f"[SKIP] Missing params CSV: {params_csv}")
        continue
    params_df = pd.read_csv(params_csv)
    params_df.columns = params_df.columns.str.strip()

    for p in PRUNING_LEVELS:
        avg_csv = os.path.join(BASE_DIR, f"p-percentage_{p}", f"batch_size_{bs}",
                               f"averaged_runs_p_{p}_bs_{bs}.csv")
        if not os.path.exists(avg_csv):
            print(f"  [SKIP] P%={p*100:5.1f}%  BS={bs:>6}  \u2014 no averaged CSV")
            continue

        row = params_df[np.isclose(params_df["P%"], p * 100)]
        if row.empty:
            print(f"  [SKIP] P%={p*100:5.1f}%  BS={bs:>6}  \u2014 no fit params row")
            continue

        cutoff_BN = float(row["cutoff_BN"].iloc[0])

        # Load full data to determine whether step was detected
        full_df = pd.read_csv(avg_csv)
        full_df.columns = full_df.columns.str.strip()
        ce_col = next((c for c in full_df.columns if c in ("Avg_CE_Test", "Avg_CE_test")), None)
        bn_col = next((c for c in full_df.columns if "Batch" in c), None)
        if ce_col is None or bn_col is None:
            print(f"  [SKIP] P%={p*100:5.1f}%  BS={bs:>6}  \u2014 unexpected columns {list(full_df.columns)}")
            continue
        full_df = full_df.dropna(subset=[ce_col, bn_col])
        max_bn  = float(full_df[bn_col].max())
        step_was_detected = cutoff_BN < max_bn

        # Scatter: only pre-step data
        avg_df = full_df[full_df[bn_col] < cutoff_BN]

        learn_BN           = float(row["learn_BN"].iloc[0])
        avg_CE_learn_at_BN = float(row["avg_CE_learn_at_BN"].iloc[0]) \
                             if "avg_CE_learn_at_BN" in row.columns else np.nan
        from_data = np.isfinite(avg_CE_learn_at_BN)

        records.append({
            "bs": bs, "p": p,
            "bn_avg":             avg_df[bn_col].values.astype(float),
            "ce_avg":             avg_df[ce_col].values.astype(float),
            "A":                  float(row["A"].iloc[0]),
            "B":                  float(row["B"].iloc[0]),
            "n":                  float(row["n"].iloc[0]),
            "CE_L":               float(row["CE_L"].iloc[0]),
            "learn_BN":           learn_BN,
            "avg_CE_learn_at_BN": avg_CE_learn_at_BN,
            "from_data":          from_data,
            "IPA":                float(row["IPA"].iloc[0]),
            "cutoff_BN":          cutoff_BN,
            "step_was_detected":  step_was_detected,
        })
        ce_str   = f"{avg_CE_learn_at_BN:.4f}" if from_data else "NaN"
        src      = "data" if from_data else "analytic fallback"
        step_tag = "[step detected]" if step_was_detected else "[no step]"
        print(f"  OK   P%={p*100:5.1f}%  BS={bs:>6}  cutoff_BN={cutoff_BN:>6.0f} {step_tag:<16}  "
              f"learn_BN={learn_BN!r}  avg_CE@BNL={ce_str}  [{src}]")

print(f"\nLoaded {len(records)} combinations.")

  OK   P%=  0.0%  BS=    64  cutoff_BN=   263 [step detected]   learn_BN=27.0  avg_CE@BNL=0.5939  [data]
  OK   P%= 10.0%  BS=    64  cutoff_BN=   273 [step detected]   learn_BN=29.0  avg_CE@BNL=0.5930  [data]
  OK   P%= 20.0%  BS=    64  cutoff_BN=   272 [step detected]   learn_BN=33.0  avg_CE@BNL=0.5840  [data]
  OK   P%= 30.0%  BS=    64  cutoff_BN=   261 [step detected]   learn_BN=34.0  avg_CE@BNL=0.6038  [data]
  OK   P%= 40.0%  BS=    64  cutoff_BN=   285 [step detected]   learn_BN=40.0  avg_CE@BNL=0.5982  [data]
  OK   P%= 50.0%  BS=    64  cutoff_BN=   322 [step detected]   learn_BN=46.0  avg_CE@BNL=0.6071  [data]
  OK   P%= 60.0%  BS=    64  cutoff_BN=   367 [step detected]   learn_BN=65.0  avg_CE@BNL=0.5806  [data]
  OK   P%= 70.0%  BS=    64  cutoff_BN=   379 [step detected]   learn_BN=83.0  avg_CE@BNL=0.5983  [data]
  OK   P%= 80.0%  BS=    64  cutoff_BN=   400 [step detected]   learn_BN=132.0  avg_CE@BNL=0.6168  [data]
  OK   P%= 82.0%  BS=    64  cutoff_BN=   439 [no step

In [12]:
# === Cell 3 — Plot averaged CE (pre-step) vs fitted curve for every (P%, BS) ===
plt.rcParams.update({"font.size": 13})

for rec in records:
    bs                 = rec["bs"]
    p                  = rec["p"]
    bn_avg             = rec["bn_avg"]
    ce_avg             = rec["ce_avg"]
    A                  = rec["A"]
    B                  = rec["B"]
    n                  = rec["n"]
    CE_L               = rec["CE_L"]
    learn_BN           = rec["learn_BN"]
    avg_CE_learn_at_BN = rec["avg_CE_learn_at_BN"]
    from_data          = rec["from_data"]
    IPA                = rec["IPA"]
    cutoff_BN          = rec["cutoff_BN"]
    step_was_detected  = rec["step_was_detected"]

    # x-axis extends to cover BNL even when beyond cutoff
    bn_end    = max(bn_avg.max() if len(bn_avg) > 0 else cutoff_BN,
                    learn_BN if np.isfinite(learn_BN) else 0) * 1.3
    bn_smooth = np.linspace(0, bn_end, 600)
    y_fit     = A + B / ((bn_smooth + 1) ** n)

    fig, ax = plt.subplots(figsize=(10, 6))

    # Averaged CE scatter (pre-step only)
    scatter_label = (r"$\overline{CE}_{test}$ (avg 100 runs, pre-step)"
                     if step_was_detected else
                     r"$\overline{CE}_{test}$ (avg 100 runs, full data)")
    ax.scatter(bn_avg, ce_avg, s=8, color="#aaaaaa", alpha=0.6, zorder=1,
               label=scatter_label)

    # Smooth fitted curve (extends beyond cutoff to show extrapolation)
    ax.plot(bn_smooth, y_fit, color="#1f77b4", linewidth=2.2, zorder=3,
            label=f"Fit (pre-step): A={A:.4f},  B={B:.4f},  n={n:.4f}")

    # Step cutoff boundary line (only when a step was actually detected)
    if step_was_detected:
        ax.axvline(cutoff_BN, color="#bbbbbb", linewidth=1.0, linestyle="-", alpha=0.6)
        ax.text(cutoff_BN + 5, CE_o - 0.05,
                f"BN={cutoff_BN:.0f}\n(step cutoff)",
                fontsize=8, color="#888888", va="top")

    # Horizontal reference lines
    ax.axhline(CE_o, color="#888888", linewidth=1.0, linestyle=":")
    ax.text(bn_end, CE_o + 0.03, f"CE_o = {CE_o:.4f}",
            ha="right", fontsize=10, color="#666666")

    ax.axhline(CE_L, color="#9467bd", linewidth=1.4, linestyle="--")
    ax.text(bn_end, CE_L + 0.03, f"CE_L = {CE_L:.4f}",
            ha="right", fontsize=10, color="#9467bd")

    ax.axhline(A, color="#2ca02c", linewidth=1.4, linestyle="--")
    ax.text(bn_end, A - 0.07, f"A = {A:.4f}  (asymptote)",
            ha="right", fontsize=10, color="#2ca02c")

    # BNL marker
    if np.isfinite(learn_BN) and learn_BN > 0:
        if from_data:
            bnl_color = "#d62728"
            ax.axvline(learn_BN, color=bnl_color, linewidth=1.4,
                       linestyle="--", alpha=0.8, zorder=4)
            ax.scatter([learn_BN], [avg_CE_learn_at_BN], s=80, color=bnl_color,
                       marker="*", zorder=5, label=f"Avg data crossing  (BN={learn_BN:.0f})")
            annot_text = f"BNL = {learn_BN:.0f}  [avg data]\nIPA = {IPA:.5f}"
        else:
            bnl_color = "#ff7f0e"
            ax.axvline(learn_BN, color=bnl_color, linewidth=1.4,
                       linestyle=":", alpha=0.8, zorder=4)
            annot_text = f"BNL = {learn_BN:.0f}  [extrapolated]\nIPA = {IPA:.5f}"

        ax.annotate(
            annot_text,
            xy=(learn_BN, CE_L),
            xytext=(learn_BN + bn_end * 0.03, CE_L + 0.15),
            fontsize=10, color=bnl_color,
            arrowprops=dict(arrowstyle="->", color=bnl_color, lw=1.0)
        )

    ax.set_xlabel("Batch Number (BN)")
    ax.set_ylabel("CE_TEST")
    ax.set_xlim(0, bn_end)
    ax.set_ylim(max(0, A - 0.15), CE_o + 0.25)
    step_tag = f"step at BN={cutoff_BN:.0f}" if step_was_detected else "no step detected"
    ax.set_title(
        f"Approach 1 (avoid-step) \u2014 Avg CE vs Fit  |  P%={p*100:.1f}%  BS={bs}\n"
        f"Cutoff: {step_tag}"
    )
    ax.legend(fontsize=10, frameon=False, loc="upper right")
    ax.grid(True, alpha=0.25)

    bs_dir  = os.path.join(OUT_DIR, f"BS_{bs}")
    os.makedirs(bs_dir, exist_ok=True)
    out_png = os.path.join(bs_dir, f"fitting_avg_plot_A_1_avoid_step_p_{p}_bs_{bs}.png")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.close(fig)
    step_label = f"step@{cutoff_BN:.0f}" if step_was_detected else "no step"
    print(f"  Saved: fitting_avg_plot_A_1_avoid_step_p_{p}_bs_{bs}.png  [{step_label}]")

print("\n[Done]")

  Saved: fitting_avg_plot_A_1_avoid_step_p_0.0_bs_64.png  [step@263]
  Saved: fitting_avg_plot_A_1_avoid_step_p_0.1_bs_64.png  [step@273]
  Saved: fitting_avg_plot_A_1_avoid_step_p_0.2_bs_64.png  [step@272]
  Saved: fitting_avg_plot_A_1_avoid_step_p_0.3_bs_64.png  [step@261]
  Saved: fitting_avg_plot_A_1_avoid_step_p_0.4_bs_64.png  [step@285]
  Saved: fitting_avg_plot_A_1_avoid_step_p_0.5_bs_64.png  [step@322]
  Saved: fitting_avg_plot_A_1_avoid_step_p_0.6_bs_64.png  [step@367]
  Saved: fitting_avg_plot_A_1_avoid_step_p_0.7_bs_64.png  [step@379]
  Saved: fitting_avg_plot_A_1_avoid_step_p_0.8_bs_64.png  [step@400]
  Saved: fitting_avg_plot_A_1_avoid_step_p_0.82_bs_64.png  [no step]
  Saved: fitting_avg_plot_A_1_avoid_step_p_0.84_bs_64.png  [step@400]
  Saved: fitting_avg_plot_A_1_avoid_step_p_0.86_bs_64.png  [step@380]
  Saved: fitting_avg_plot_A_1_avoid_step_p_0.88_bs_64.png  [no step]
  Saved: fitting_avg_plot_A_1_avoid_step_p_0.9_bs_64.png  [step@400]
  Saved: fitting_avg_plot_A_1_av